In [3]:
!pip install -q -U pypdf sentence-transformers chromadb groq

In [6]:
from google.colab import userdata, files
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb
import os

In [11]:
from google.colab import userdata

# Get Groq API key from Colab Secrets
GROQ_API_KEY = userdata.get("aaditi21")

# Check whether key was loaded
if not GROQ_API_KEY:
    raise ValueError("API key not found. Please create a Colab Secret named 'aadiiti21'.")

print("API key loaded successfully!")

API key loaded successfully!


In [33]:
# Create Groq client
client = Groq(
    api_key=GROQ_API_KEY
)

# Groq model
MODEL_NAME = "openai/gpt-oss-20b"

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [34]:
print("Please upload your PDF file:")

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print("PDF uploaded successfully!")
print(pdf_filename)

Please upload your PDF file:


Saving DSCC.pdf to DSCC (1).pdf
PDF uploaded successfully!
DSCC (1).pdf


In [35]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)

pdf_text = ""

for page_number, page in enumerate(reader.pages):
    text = page.extract_text()

    if text:
        pdf_text += text + "\n"

print("PDF text extracted successfully!")
print("Number of characters:", len(pdf_text))

PDF text extracted successfully!
Number of characters: 140150


In [36]:
print(pdf_text[:2000])

 
 
UNIT
 
1
 
 
What
 
is
 
a
 
Distributed
 
System?
 
Explain
 
its
 
Architecture
 
1.
 
Definition
 
of
 
Distributed
 
System
 
A
 
distributed
 
system
 
is
 
a
 
group
 
of
 
independent
 
computers
 
connected
 
through
 
a
 
network
 
that
 
work
 
together
 
as
 
a
 
single
 
system
.
 
Each
 
computer
 
in
 
the
 
distributed
 
system
 
is
 
called
 
a
 
node
 
or
 
processor
.
 
Every
 
processor
 
has
 
its
 
own
 
local
 
memory
 
and
 
other
 
resources.
 
The
 
processors
 
communicate
 
with
 
each
 
other
 
through
 
a
 
network
 
by
 
passing
 
messages
.
 
In
 
simple
 
words,
 
instead
 
of
 
doing
 
all
 
the
 
work
 
on
 
one
 
computer,
 
the
 
work
 
is
 
divided
 
among
 
multiple
 
computers
 
that
 
communicate
 
and
 
cooperate
 
with
 
each
 
other.
 
Example
 
Some
 
common
 
examples
 
are:
 
●
 
Google
 
Search
 
●
 
Online
 
Banking
 
●
 
Google
 
Drive
 
/
 
Cloud
 
Storage
 
●
 
A
 
nationalized
 
bank
 
having
 
multiple
 
branches
 
connected
 
th

In [21]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [37]:
chunks = chunk_text(pdf_text)

print("Total PDF chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print("\n--- CHUNK", i + 1, "---")
    print(chunk[:500])

Total PDF chunks: 176

--- CHUNK 1 ---
UNIT
 
1
 
 
What
 
is
 
a
 
Distributed
 
System?
 
Explain
 
its
 
Architecture
 
1.
 
Definition
 
of
 
Distributed
 
System
 
A
 
distributed
 
system
 
is
 
a
 
group
 
of
 
independent
 
computers
 
connected
 
through
 
a
 
network
 
that
 
work
 
together
 
as
 
a
 
single
 
system
.
 
Each
 
computer
 
in
 
the
 
distributed
 
system
 
is
 
called
 
a
 
node
 
or
 
processor
.
 
Every
 
processor
 
has
 
its
 
own
 
local
 
memory
 
and
 
other
 
resources.
 
The
 
processors
 
communic

--- CHUNK 2 ---
Some
 
common
 
examples
 
are:
 
●
 
Google
 
Search
 
●
 
Online
 
Banking
 
●
 
Google
 
Drive
 
/
 
Cloud
 
Storage
 
●
 
A
 
nationalized
 
bank
 
having
 
multiple
 
branches
 
connected
 
through
 
a
 
network
 
These
 
systems
 
use
 
multiple
 
computers
 
or
 
locations
 
but
 
provide
 
services
 
as
 
one
 
overall
 
system.
 
2.
 
Example
 
of
 
a
 
Distributed
 
System
 
A
 
good
 
example
 
is
 
a
 
banking
 
system
.
 
Supp

In [38]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [39]:
embeddings = embedder.encode(chunks).tolist()

print("Embeddings created successfully!")
print("Number of embeddings:", len(embeddings))

Embeddings created successfully!
Number of embeddings: 176


In [40]:
# Create ChromaDB client
chroma_client = chromadb.Client()

# Create collection
collection = chroma_client.get_or_create_collection(
    name="pdf_chatbot"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [41]:
# Create unique IDs
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=ids
)

print("PDF chunks stored in ChromaDB successfully!")
print("Total documents in collection:", collection.count())

PDF chunks stored in ChromaDB successfully!
Total documents in collection: 176


In [42]:
def retrieve_pdf_context(query: str, top_k: int = 3) -> list[str]:

    # Convert user's question into embedding
    query_embedding = embedder.encode([query]).tolist()

    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    # Return retrieved PDF documents
    return results["documents"][0]

In [43]:
def ask_pdf(query: str):

    # Retrieve relevant PDF passages
    context_passages = retrieve_pdf_context(query, top_k=3)

    # Combine retrieved passages
    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    # Create prompt
    prompt = f"""
You are an intelligent document analysis assistant.

Answer the question using ONLY the information provided
in the PDF context below.

If the information is not contained within the provided
context, clearly state:

"I cannot find the answer in the provided PDF."

Do not make up information.

PDF Context:
{context_str}

Question:
{query}

Answer:
"""

    # Send request to Groq
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    # Get answer
    answer = response.choices[0].message.content

    return answer, context_passages

In [30]:
def ask_pdf(query: str):

    # Retrieve relevant PDF passages
    context_passages = retrieve_pdf_context(query, top_k=3)

    # Combine retrieved passages
    context_str = "\n".join(
        f"- {p}" for p in context_passages
    )

    # Prompt
    prompt = f"""
You are an intelligent PDF document analysis assistant.

Answer the user's question using ONLY the information
provided in the PDF context below.

If the information is not available in the PDF context,
say:
"I cannot find the answer in the provided PDF."

For summary questions, give a clear and concise summary
based on the retrieved PDF content.

Do not make up information.

PDF Context:
{context_str}

Question:
{query}

Answer:
"""

    # Groq API call
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1000
    )

    # Get answer
    answer = response.choices[0].message.content

    return answer, context_passages

In [44]:
question = input("Ask a question about your PDF: ")

answer, context = ask_pdf(question)

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)

print(answer)

Ask a question about your PDF: What is this PDF about? Give me a summary of it.

ANSWER
The PDF contains a brief overview of several distributed‑systems concepts, mainly focusing on mutual‑exclusion algorithms and the client‑server communication model. It describes:

1. **Master Algorithm** – a central master process that grants access to a critical section, using three messages (Request, Grant, Release) to coordinate processes.
2. **Distributed Algorithm** – a decentralized approach where each process timestamps its request, broadcasts it to all others, and the process with the lowest timestamp gains priority; after finishing, it sends an OK message to the next waiting process.
3. **Token Ring Algorithm** – processes arranged in a logical ring that pass a special token; a request is sent to the server, which processes it and replies to the client.
4. **Client–Server Model Diagram** – a description of the typical request‑reply flow between a client and a server, including network routi

In [45]:
print("\n" + "=" * 60)
print("RETRIEVED PDF SNIPPETS")
print("=" * 60)

for i, snippet in enumerate(context, 1):
    print(f"\n--- Snippet {i} ---")
    print(snippet[:1500])


RETRIEVED PDF SNIPPETS

--- Snippet 1 ---
master
 
maintains
 
a
 
list
 
of
 
processes
 
requesting
 
the
 
critical
 
section.
 
●
 
It
 
grants
 
permission
 
to
 
processes
 
in
 
some
 
order.
 
●
 
Three
 
messages
 
are
 
used
 
for
 
one
 
critical-section
 
operation:
 
○
 
Request
 
(R)
 
○
 
Grant
 
(G)
 
○
 
Release
 
(R)
 
Process
 
→
 
Request
 
→
 
Master
 
Process
 
←
 
Grant
   
←
 
Master
 
Process
 
→
 
Release
 
→
 
Master
 
 
2.
 
Distributed
 
Algorithm
 
●
 
There
 
is
 
no
 
central
 
master
.
 
●
 
When
 
a
 
process
 
wants
 
to
 
enter
 
the
 
critical
 
section,
 
it
 
creates
 
a
 
timestamped
 
request
.
 
●
 
It
 
sends
 
the
 
request
 
to
 
all
 
other
 
processes,
 
including
 
itself.
 
●
 
The
 
process
 
with
 
the
 
lowest
 
timestamp
 
gets
 
priority.
 
●
 
After
 
finishing,
 
it
 
sends
 
an
 
OK
 
message
 
to
 
the
 
waiting
 
process.
 
 
3.
 
Token
 
Ring
 
Algorithm
 
●
 
Processes
 
are
 
arranged
 
in
 
a
 
logical
 
ring
.
 
●
 
A
 
s

In [46]:
print("\n" + "=" * 60)
print("          PDF CHATBOT READY!")
print("=" * 60)

print("Ask questions about your PDF.")
print("Type 'exit', 'quit' or 'q' to stop.")
print("=" * 60)


while True:

    user_query = input("\nAsk a question about your PDF: ")

    # Exit chatbot
    if user_query.lower() in ["exit", "quit", "q"]:
        print("\nExiting PDF Chatbot. Goodbye!")
        break

    # Ignore empty input
    if not user_query.strip():
        continue

    # Get answer from PDF
    answer, context = ask_pdf(user_query)

    # --------------------------------------------------------
    # RETRIEVED PDF SNIPPETS
    # --------------------------------------------------------

    print("\n" + "-" * 60)
    print("RETRIEVED PDF SNIPPETS")
    print("-" * 60)

    for i, snippet in enumerate(context, 1):
        print(f"\n[{i}] {snippet[:1500]}")

    # --------------------------------------------------------
    # CHATBOT RESPONSE
    # --------------------------------------------------------

    print("\n" + "-" * 60)
    print("CHATBOT RESPONSE")
    print("-" * 60)

    print(answer)

    print("-" * 60)


          PDF CHATBOT READY!
Ask questions about your PDF.
Type 'exit', 'quit' or 'q' to stop.

Ask a question about your PDF: Give summary of this pdf in 200 words

------------------------------------------------------------
RETRIEVED PDF SNIPPETS
------------------------------------------------------------

[1] ration
 
The
 
server
 
stub
 
prepares:
 
₹50,000
 
for
 
transmission.
 
Step
 
8
 
—
 
Result
 
transmission
 
The
 
server
 
sends
 
the
 
result:
 
Server
 
→
 
Network
 
→
 
Client
 
Step
 
9
 
—
 
Client
 
receives
 
result
 
The
 
client
 
stub
 
receives
 
and
 
unmarshals
 
the
 
result.
 
Step
 
10
 
—
 
Client
 
gets
 
result
 
The
 
client
 
program
 
receives:
 
₹50,000
 
Thus,
 
the
 
client
 
can
 
use
 
the
 
remote
 
service
 
without
 
directly
 
handling
 
the
 
underlying
 
network
 
communication.
 
 
6.
 
Marshalling
 
and
 
Unmarshalling
 
—
 
Very
 
Important
 
These
 
two
 
terms
 
are
 
frequently
 
asked
 
in
 
exams.
 
Marshalling
 
Marshalling
 